# CropGuard - ConvNeXt Challenger + A/B Test

Trains the challenger, then runs the **first real statistical comparison** against the
deployed ResNet50 baseline.

Runs on Lightning AI or Colab - the setup cell detects which. Attach a GPU.

| Step | Time (T4) |
|---|---|
| Setup + data (cached if section 3 of notebook 01 already ran) | ~20 min |
| **Train ConvNeXt-Tiny** | **~2-3 hrs** |
| Export + predict both models | ~25 min |
| A/B comparison | seconds |

---

## Read this before quoting the result

The two configs differ in **five** ways at once:

| | Baseline | Challenger |
|---|---|---|
| Architecture | ResNet50 | ConvNeXt-Tiny |
| Augmentation | medium | heavy |
| Epochs | 12 | 30 |
| Learning rate | 3e-4 | 1e-4 |
| Weight decay | 1e-4 | 5e-2 |

So this answers **"which configuration should ship?"** - a legitimate and useful question.
It does **not** answer "is ConvNeXt better than ResNet50", because five variables moved
together and no single one can be credited. Calling this an architecture comparison would be
an overclaim, and it is exactly the kind an interviewer will probe.

Isolating any one factor needs a controlled ablation - same architecture, one variable
changed - which is Week 2 work and is not built.

**Also expect this to come out null.** The baseline is at 99.11% on a saturated benchmark,
leaving 0.9% of headroom. A framework that declines to promote a challenger is doing its job;
that is a better result to talk about than a win you cannot defend.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

if Path('/teamspace/studios/this_studio').exists():
    ENV, BASE = 'lightning', '/teamspace/studios/this_studio'
elif Path('/content').exists():
    ENV, BASE = 'colab', '/content'
else:
    ENV, BASE = 'local', str(Path.home())

REPO = f'{BASE}/CropGuard'
DATA = f'{BASE}/cropguard-data'
SRC  = f'{REPO}/src'
print('environment:', ENV, '| repo:', REPO)

In [ ]:
if Path(REPO + '/.git').exists():
    !git -C {REPO} pull -q
else:
    !rm -rf {REPO}
    !git clone -q https://github.com/abhinav7289A/CropGuard.git {REPO}

os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip install -q pytorch-lightning timm torchmetrics wandb onnx onnxruntime huggingface_hub scikit-learn scipy tqdm pyyaml

In [ ]:
import importlib

if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
os.environ['PYTHONPATH'] = SRC
os.environ['CROPGUARD_DATA_DIR'] = DATA
os.environ['PYTHONIOENCODING'] = 'utf-8'

import cropguard, torch
print('cropguard', cropguard.__version__, '| torch', torch.__version__,
      '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'No GPU - attach one before training'

## 2. Data

Skipped if already present. The split hash **must** match: the A/B test is paired, so both
models have to be scored on the identical holdout in the identical order. A different split
makes the comparison meaningless rather than merely noisy.

In [ ]:
if Path(DATA + '/plantvillage/manifest.json').exists():
    import json
    m = json.load(open(DATA + '/plantvillage/manifest.json'))
    print(f"dataset present: {m['num_images']:,} images / {m['num_classes']} classes")
else:
    !python -m cropguard.data.download --config configs/base.yaml
    !python -m cropguard.data.validate --config configs/base.yaml

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml

import hashlib, json
EXPECTED = '9764d8f2eb2046e9ba91a138e21d472bd6a9e512232431b7d62d252c6ea8efba'
actual = hashlib.sha256(open(DATA + '/splits.json','rb').read()).hexdigest()
print('split hash:', 'MATCH' if actual == EXPECTED else 'MISMATCH -> ' + actual)
assert actual == EXPECTED, 'Split differs - the paired A/B test would be invalid'

## 3. Weights & Biases (optional)

Leave `WANDB_ENTITY` empty unless logging to a team. If W&B fails, training falls back to
CSV and carries on.

In [ ]:
WANDB_API_KEY = ''   # <- paste your key
WANDB_ENTITY  = ''   # <- leave empty for a personal account

os.environ.pop('WANDB_ENTITY', None)
if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.pop('WANDB_MODE', None)
    import wandb; wandb.login(key=WANDB_API_KEY)
    print('W&B enabled')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled - CSV logging only')

## 4. Checkpoint persistence

ConvNeXt at 30 epochs is a 2-3 hour run - long enough that a reclaimed session is likely.
Checkpoints are written every epoch and `--resume` picks up from `last.ckpt`.

In [ ]:
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/cropguard/checkpoints
    !ln -sfn /content/drive/MyDrive/cropguard/checkpoints {REPO}/checkpoints
    print('checkpoints -> Google Drive')
else:
    os.makedirs(f'{REPO}/checkpoints', exist_ok=True)
    print('checkpoints ->', f'{REPO}/checkpoints', '(persistent)')

## 5. Train the challenger

ConvNeXt-Tiny, heavy augmentation, 30 epochs with early stopping (patience 5). Expect it to
stop well before 30 - the baseline hit 0.9926 val macro-F1 in 12.

**If the session dies:** re-run cells 1-4, then append
`--resume checkpoints/convnext-tiny-challenger/last.ckpt`.

**If you hit CUDA OOM:** ConvNeXt uses more activation memory than ResNet50 at the same
batch size. Drop `batch_size` to 32 in the config cell and say so when reporting results.

In [ ]:
workers = min(8, max(2, (os.cpu_count() or 2) - 1))
config = f'''# Generated for this runtime.
extends: convnext_tiny.yaml

data:
  num_workers: {workers}
'''
Path('configs/runtime_convnext.yaml').write_text(config)
print(config)

In [ ]:
# 30-second smoke test before committing to a multi-hour run.
!python -m cropguard.training.train --config configs/runtime_convnext.yaml --fast-dev-run

In [ ]:
!python -m cropguard.training.train --config configs/runtime_convnext.yaml

## 6. Export the challenger

Gated by the PyTorch parity check - a silently wrong graph would corrupt the A/B result
rather than fail it.

In [ ]:
import glob
ckpts = sorted(glob.glob('checkpoints/convnext-tiny-challenger/best-*.ckpt'))
assert ckpts, 'No checkpoint - did training finish?'
CKPT = ckpts[-1]
print('exporting', CKPT)

!python -m cropguard.serving.onnx_export --ckpt "{CKPT}" --out models/challenger.onnx
!ls -la models/

## 7. Predictions from BOTH models on the same holdout

McNemar and the bootstrap are **paired** tests: they compare the two models image by image,
which is what makes them far more sensitive than comparing two accuracy numbers. That
requires both models scored on the identical images in the identical order.

The baseline comes from HF Hub rather than being retrained, so this is the exact artifact
currently serving production traffic.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

baseline = hf_hub_download('XiElonMAsk/cropguard-models', 'cropguard.onnx')
shutil.copy(baseline, 'models/baseline.onnx')
print('baseline pulled from HF Hub ->', Path('models/baseline.onnx').stat().st_size / 1e6, 'MB')

In [ ]:
# ~12 min each on a T4. Both write logits as well as probabilities.
!python -m cropguard.evaluation.predict --model models/baseline.onnx \
    --split test --out artifacts/preds_baseline.npz --model-version resnet50-baseline
!python -m cropguard.evaluation.predict --model models/challenger.onnx \
    --split test --out artifacts/preds_challenger.npz --model-version convnext-tiny-challenger

## 8. The A/B test

McNemar on per-image correctness, a paired bootstrap CI on the accuracy difference, a
per-class t-test with Cohen's d, and a power analysis.

Promotion requires **all three**: McNemar significant, in the right direction, and a
bootstrap CI excluding zero. A significant p-value alone would happily promote a model that
is significantly *worse* - McNemar is two-sided.

`compare` exits non-zero when the challenger does not win, so CI can gate on it.

In [ ]:
!python -m cropguard.evaluation.compare \
    --baseline artifacts/preds_baseline.npz \
    --challenger artifacts/preds_challenger.npz \
    --out artifacts/ab_comparison.json
print()
print('exit code 0 = challenger wins, 1 = no significant improvement (both are results)')

In [ ]:
# Headline numbers side by side.
import json
from sklearn.metrics import f1_score
from cropguard.evaluation.predict import load_predictions

b = load_predictions('artifacts/preds_baseline.npz')
c = load_predictions('artifacts/preds_challenger.npz')

print(f"{'':12s} {'accuracy':>10s} {'macro-F1':>10s}")
for name, d in (('baseline', b), ('challenger', c)):
    acc = d['correct'].mean()
    f1 = f1_score(d['labels'], d['predictions'], average='macro')
    print(f'{name:12s} {acc:10.4f} {f1:10.4f}')
print()
print(f"accuracy delta: {c['correct'].mean() - b['correct'].mean():+.4f}")

### Calibration of the challenger

Worth checking separately: the challenger uses the same label smoothing, so it should show
the same ~0.90 confidence ceiling and need its own temperature. **A temperature fitted for
one model does not transfer to another** - it is a property of that model's logits.

In [ ]:
# Needs validation predictions for the challenger. ~4 min.
!python -m cropguard.evaluation.predict --model models/challenger.onnx \
    --split val --out artifacts/preds_challenger_val.npz --model-version convnext-val

!python -m cropguard.evaluation.calibrate \
    --val artifacts/preds_challenger_val.npz \
    --test artifacts/preds_challenger.npz \
    --out artifacts/challenger_calibration.json

## 9. Where the two models disagree

McNemar only uses the discordant pairs, so this is the evidence the whole test rests on.
If they disagree on very few images, the comparison is thin regardless of the p-value.

In [ ]:
import numpy as np

only_b = (~b['correct']) & c['correct']   # challenger fixed these
only_a = b['correct'] & (~c['correct'])   # challenger broke these
print(f'challenger fixed : {only_b.sum()}')
print(f'challenger broke : {only_a.sum()}')
print(f'discordant total : {only_b.sum() + only_a.sum()} of {len(b["labels"])}')
print()

classes = json.load(open('configs/classes.json'))
for label, mask in (('FIXED by challenger', only_b), ('BROKEN by challenger', only_a)):
    idx = np.flatnonzero(mask)[:5]
    print(label)
    for i in idx:
        print(f"   {classes[b['labels'][i]][:38]:38s} "
              f"base->{classes[b['predictions'][i]][:26]:26s} "
              f"chal->{classes[c['predictions'][i]][:26]}")
    print()

## 10. Save the artifacts

The prediction files and the comparison report are what get written up. Keep them even if
the challenger lost - a measured null is the result, not a failed experiment.

In [ ]:
import shutil, os

dest = '/content/drive/MyDrive/cropguard/artifacts' if ENV == 'colab' else f'{BASE}/artifacts'
os.makedirs(dest, exist_ok=True)
for pattern in ('artifacts/*.npz', 'artifacts/*.json', 'models/challenger.onnx'):
    for f in glob.glob(pattern):
        shutil.copy(f, dest)
        print('saved', f)
print('->', dest)

---
### Reporting this honestly

Whatever the outcome:

- **If the challenger wins:** say *this configuration* beat *that configuration*. Five
  variables differ, so no single one can be credited.
- **If it does not:** that is a result. "I built the framework, ran it, and it declined to
  promote the challenger" demonstrates the discipline the project is about. Report the
  effect size and the power - a null from an underpowered test means *could not tell*, not
  *no difference*.
- **Either way**, quote the discordant-pair count. McNemar rests entirely on it.